# CampusFit AI — Dataset Preprocessing & Baseline Models
**CSL236 practical notebook**  
Team number: __________  
Members / roll numbers: __________  

> Both datasets in this repository are synthetic educational demo data generated with a fixed seed. They are not real participant or clinical records. This notebook demonstrates the required ML workflow; the inclusive labels are curated categories, not medical recommendations.

## 1. Imports and dataset loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

ROOT = Path.cwd()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT = ROOT.parent
general = pd.read_csv(ROOT/'data/general_fitness.csv')
inclusive = pd.read_csv(ROOT/'data/inclusive_activity.csv')
print('General:', general.shape, 'Inclusive:', inclusive.shape)

## 2. Data understanding

In [ ]:
display(general.head())
display(general.describe(include='all').T)
display(general.dtypes.to_frame('dtype'))
missing = pd.DataFrame({'missing_count':general.isna().sum(), 'missing_percent':(general.isna().mean()*100).round(2)})
display(missing)
numeric_cols = general.select_dtypes(include=np.number).columns.tolist()
categorical_cols = general.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric:', numeric_cols)
print('Categorical:', categorical_cols)

## 3. Missing-value strategy

For the general dataset, numerical features use median imputation because it is robust to skew and outliers; categorical features use most-frequent imputation to retain rows. The imputer is fitted only on the training partition below. The target column is not imputed; generated target labels are present for every row.

## 4. Train/test split BEFORE fitting preprocessing (leakage prevention)

In [ ]:
target = 'recommended_intensity'
X = general.drop(columns=[target]); y = general[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Encoding, scaling and model training

Nominal categories (activity level and fitness goal) are one-hot encoded; there are no ordinal text columns in this dataset, so ordinal encoding is not applicable. Numeric features are median-imputed and standardized for a consistent preprocessing pipeline. The Random Forest is a tree ensemble and does not strictly require scaling; scaling is included here to demonstrate a reusable, leakage-safe numeric preprocessing pipeline.

In [ ]:
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric_pipe, num), ('cat', categorical_pipe, cat)])
model = Pipeline([('preprocess', preprocess), ('select', SelectKBest(f_classif, k=10)), ('classifier', RandomForestClassifier(n_estimators=180, random_state=42, class_weight='balanced'))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred,zero_division=0))
print('Original input features:', X.shape[1])
print('Transformed features before selection:', model.named_steps['preprocess'].transform(X_train).shape[1])
print('Features retained after selection:', model.named_steps['select'].get_support().sum())

## 6. Binning, outliers and dimensionality reduction

Binning: age could be grouped into age bands for descriptive reporting, but the model retains continuous age because the tree model can learn thresholds and no domain-specific clinical boundaries are justified. We therefore do not replace age with arbitrary bins. Outliers: inspect IQR; values may be plausible human variation, so do not delete automatically. Feature selection: SelectKBest is included with k=all as a transparent baseline (no features removed); this preserves all transformed features. A future experiment can compare smaller k values and report the resulting count.

In [ ]:
for col in ['age','height_m','weight_kg','bmi','sleep_hours','weekly_exercise_days']:
    q1, q3 = general[col].quantile([.25,.75]); iqr=q3-q1
    lo, hi=q1-1.5*iqr, q3+1.5*iqr
    print(col, 'IQR bounds:', round(lo,2), round(hi,2), 'flagged:', int(((general[col]<lo)|(general[col]>hi)).sum()))
# Example age bin for descriptive analysis only; original feature remains in model data.
general['age_band_demo'] = pd.cut(general.age, bins=[0,25,40,60,100], labels=['18-25','26-40','41-60','60+'])
display(general['age_band_demo'].value_counts(dropna=False))

## 7. Inclusive dataset: separate pipeline

In [ ]:
display(inclusive.head())
display(pd.DataFrame({'missing_count':inclusive.isna().sum(), 'missing_percent':(inclusive.isna().mean()*100).round(2)}))
target2='suggested_activity'; X2=inclusive.drop(columns=[target2]); y2=inclusive[target2]
X2_train,X2_test,y2_train,y2_test=train_test_split(X2,y2,test_size=.2,random_state=42,stratify=y2)
num2=X2.select_dtypes(include=np.number).columns.tolist(); cat2=X2.select_dtypes(exclude=np.number).columns.tolist()
prep2=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),num2),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat2)])
model2=Pipeline([('preprocess',prep2),('select',SelectKBest(f_classif,k='all')),('classifier',DecisionTreeClassifier(max_depth=6,min_samples_leaf=5,class_weight='balanced',random_state=42))])
model2.fit(X2_train,y2_train); pred2=model2.predict(X2_test)
print('Accuracy:',round(accuracy_score(y2_test,pred2),4)); print(classification_report(y2_test,pred2,zero_division=0))
print('Original input features:',X2.shape[1], 'Transformed before selection:',model2.named_steps['preprocess'].transform(X2_train).shape[1], 'Retained after selection:',model2.named_steps['select'].get_support().sum())

## 8. Visuals and observations

In [ ]:
general.recommended_intensity.value_counts().plot(kind='bar', title='General fitness target distribution', xlabel='Intensity category', ylabel='Rows'); plt.tight_layout(); plt.show()
inclusive.suggested_activity.value_counts().plot(kind='bar', title='Inclusive activity target distribution', xlabel='Suggested activity', ylabel='Rows'); plt.tight_layout(); plt.show()

## 9. Conclusion and limitations

The notebook demonstrates two separate supervised classification pipelines and the preprocessing decisions. Accuracy is a basic baseline metric, not proof of fitness suitability. The labels originate from synthetic rules and the datasets are not representative of real users. No clinical, rehabilitation or disability-related claims should be made. Cite the synthetic-data generator in the report and add any external dataset citations if you replace these CSVs.